# P1 — ¿Qué es el over-squashing?

Una GNN actualiza cada nodo mezclando a sus vecinos, **un salto a la vez**. Para alcanzar información que está
a `g` saltos, necesitas `g` capas. El problema: si muchos caminos confluyen en un nodo, una enorme cantidad de
mensajes debe empacarse en el único **vector de tamaño fijo** de ese nodo. Cuando hay más para empacar de lo
que el vector puede guardar, se pierde información — eso es el **over-squashing**.

**5 figuras:** el grafo cuello de botella · un mensaje saltando capa por capa · el *vector que se desborda* · la
explosión de mensajes · de dónde vienen los mensajes.

In [ ]:
import torch, numpy as np
from oversquash import viz
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators
K, M = 4, 3

## Figura 1 — el cuello de botella

Las fuentes (azul) pasan por un medio estrecho (amarillo) hacia un objetivo (verde). **Todo** camino cruza la
parte estrecha — ese es el cuello de botella.

In [ ]:
data = make_bottleneck_graph(K, M, 2, torch.Generator().manual_seed(0))
viz.draw_bottleneck(data, K, M, title='fuentes -> cuello de botella -> objetivo')

## Figura 2 — un mensaje viaja un salto por capa

La información se mueve paso a paso. Llegar de una fuente lejana al objetivo toma tantas capas como largo sea
el camino. Mira el mensaje resaltado avanzar un salto en cada panel.

In [ ]:
dia = [(0,1),(0,2),(1,3),(2,3)]; pos = {0:(0,0),1:(1,1),2:(1,-1),3:(2,0)}
viz.explain_message_hops(dia, pos, path=[(0,1),(1,3)], n_nodes=4, src=0, dst=3,
                         title='un mensaje, un salto por capa', step_fmt='capa {s}')

## Figura 3 — el vector que se desborda (esto *es* el over-squashing)

El vector del objetivo tiene una **capacidad fija**. A medida que el grafo se hace más profundo, el número de
mensajes que debe absorber crece rápido. Una vez que superan la capacidad, el vector se desborda y se pierde
información. Esta única imagen es todo el problema.

In [ ]:
msgs = []
for d in [1,2,3,4]:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, _ = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    t = int(dd.root_mask.nonzero()[0]); msgs.append(int(raw[d][:, t].sum()))
viz.explain_overflow([(d, m) for d, m in zip([1,2,3,4], msgs)], capacity=12,
                     title='vector de capacidad fija vs. mensajes crecientes',
                     panel_fmt='profundidad {d}\n{m} mensajes', lost_fmt='¡{n} perdidos!')

## Figura 4 — qué tan rápido explotan los mensajes

Los mismos conteos en escala logarítmica: el número de mensajes crece como `K·M^profundidad` —
exponencialmente con la distancia.

In [ ]:
viz.plot_message_explosion([1,2,3,4], msgs, 'mensajes apretados en un vector', 'profundidad', 'mensajes (log)')

## Figura 5 — de dónde vienen los mensajes

La matriz de multiplicidad en el rango más profundo: fila = objetivo, columna = fuente, color = número de
recorridos. La columna brillante hacia el objetivo es la acumulación que se aprieta. **Siguiente (P2): cómo
contar estos caminos exactamente.**

In [ ]:
deep = make_bottleneck_graph(K, M, 3, torch.Generator().manual_seed(0))
raw, _ = walk_operators(deep.edge_index.numpy(), deep.num_nodes, max_length=4)
viz.plot_multiplicity_heatmap(raw[3], 'multiplicidad de recorridos en rango 4 (A^4)')